# Regime-Aware Trading Framework — Demo Research Notebook

This notebook demonstrates the full research pipeline end-to-end:

1. Data loading (yfinance / CSV fallback)
2. Feature engineering
3. Regime detection with visualisation
4. News filtering
5. Strategy signal generation
6. Backtesting with metrics
7. Parameter optimization
8. Walk-forward validation
9. Strategy selection pipeline
10. Signal generation and MT5 output

**No API keys required** — uses yfinance (free) by default.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('')))

from utils.env_loader import load_env
from utils.logger import configure_logging
load_env()
configure_logging()

import warnings
warnings.filterwarnings('ignore')
print('Framework ready.')

## 1. Load Market Data

In [ ]:
from data.market_data import MarketDataService

svc = MarketDataService(provider='yfinance')

# Load EURUSD hourly data
df_eur = svc.get('EURUSD', timeframe='1h', start='2023-01-01')
df_jpy = svc.get('USDJPY', timeframe='1h', start='2023-01-01')
df_xau = svc.get('XAUUSD', timeframe='1h', start='2023-01-01')

print(f'EURUSD: {len(df_eur)} bars  |  last close: {df_eur["close"].iloc[-1]:.5f}')
print(f'USDJPY: {len(df_jpy)} bars  |  last close: {df_jpy["close"].iloc[-1]:.3f}')
print(f'XAUUSD: {len(df_xau)} bars  |  last close: {df_xau["close"].iloc[-1]:.2f}')
df_eur.tail(3)

## 2. Feature Engineering

In [ ]:
from features.regime_features import add_regime_features, get_feature_columns
from features.alpha_features import add_alpha_features

df_eur = add_regime_features(df_eur)
df_eur = add_alpha_features(df_eur)

df_jpy = add_regime_features(df_jpy)
df_xau = add_regime_features(df_xau)

print('Feature columns:', get_feature_columns())
print(f'\nEURUSD after features: {df_eur.shape}')
df_eur[['close', 'atr', 'adx', 'rsi', 'bb_width', 'vol_ratio']].tail(5)

## 3. Regime Detection

In [ ]:
from regimes.regime_service import RegimeService

regime_svc = RegimeService(
    use_ensemble=True,
    hmm_states=4,
    smooth_window=5,
)

print('Fitting regime model on EURUSD...')
regime_svc.fit(df_eur)

regime_df = regime_svc.detect(df_eur, asset='EURUSD')
print(f'\nRegime distribution:')
print(regime_df['smoothed_regime'].value_counts())

# Latest regime
latest = regime_svc.latest(df_eur, asset='EURUSD')
print(f'\nCurrent regime: {latest.predicted_regime} (smooth: {latest.smoothed_regime})')
print(f'Confidence:     {latest.confidence:.2%}')
print(f'Tradable:       {latest.is_tradable}')

## 4. Visualise Regime Timeline

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from utils.plotting import plot_regime_timeline, plot_regime_probabilities

# Regime timeline
fig = plot_regime_timeline(
    price=df_eur['close'],
    regimes=regime_df['smoothed_regime'],
    title='EURUSD H1 — Regime Timeline',
)
plt.show()

# Probability stacked area
prob_cols = [c for c in regime_df.columns if c.startswith('p_')]
prob_df = regime_df[prob_cols].copy()
prob_df.columns = [c[2:] for c in prob_df.columns]
fig2 = plot_regime_probabilities(prob_df, title='EURUSD — Regime Probabilities')
plt.show()

## 5. News Filtering

In [ ]:
from filters.tradability_filter import TradabilityFilter

tf = TradabilityFilter(
    sessions=['london', 'new_york'],
    block_high_impact=True,
    block_medium_impact=False,
)

# Note: load_news() requires a news CSV or API key.
# Here we only apply session + liquidity filters (no news data needed).
tradable = tf.is_tradable(df_eur, apply_news=False)
summary  = tf.summary(df_eur)

print('Filter summary:')
for k, v in summary.items():
    print(f'  {k}: {v}')

## 6. Strategy Signal Generation

In [ ]:
from strategies.trend_breakout import TrendBreakoutStrategy
from strategies.mean_reversion import MeanReversionStrategy

# Trend breakout
tb = TrendBreakoutStrategy()
signals_long  = tb.generate_signals(df_eur, side='long')
signals_short = tb.generate_signals(df_eur, side='short')

n_long  = (signals_long['signal']  == 1).sum()
n_short = (signals_short['signal'] == -1).sum()
print(f'TrendBreakout LONG signals:  {n_long}')
print(f'TrendBreakout SHORT signals: {n_short}')

# Mean reversion
mr = MeanReversionStrategy()
signals_mr = mr.generate_signals(df_eur, side='long')
print(f'MeanReversion LONG signals:  {(signals_mr["signal"]==1).sum()}')

## 7. Vectorized Backtest

In [ ]:
from optimization.optimizer import Optimizer
from validation.metrics import compute_trade_metrics, compute_equity_metrics

tb = TrendBreakoutStrategy()
opt = Optimizer(tb, df_eur, method='grid')

# Simulate with default parameters
signals = tb.generate_signals(df_eur, side='long')
trades, equity = opt._simulate(signals, side='long')

print(f'Trades: {len(trades)}')
if not trades.empty:
    tm = compute_trade_metrics(trades)
    em = compute_equity_metrics(equity)
    
    print(f'\nTrade Metrics:')
    for k, v in tm.items():
        print(f'  {k:30s}: {v}')
    
    print(f'\nEquity Metrics:')
    for k, v in em.items():
        print(f'  {k:30s}: {v}')

## 8. Equity Curve Visualisation

In [ ]:
from utils.plotting import plot_equity_curve, plot_trade_distribution

if not trades.empty:
    fig = plot_equity_curve(equity, title='EURUSD Trend Breakout — Long Only')
    plt.show()
    
    fig2 = plot_trade_distribution(trades, title='Trade P&L Distribution')
    plt.show()
    
    print('Monthly returns:')
    from utils.plotting import plot_monthly_returns
    fig3 = plot_monthly_returns(equity)
    plt.show()
else:
    print('No trades to plot — try adjusting strategy parameters.')

## 9. Parameter Optimization (Optuna)

In [ ]:
from optimization.optimizer import Optimizer
from strategies.trend_breakout import TrendBreakoutStrategy

strategy = TrendBreakoutStrategy()
optimizer = Optimizer(
    strategy=strategy,
    df=df_eur,
    method='optuna',
    n_trials=50,   # use 200+ for production
    min_trades=15,
)

print('Running Optuna optimization (50 trials)...')
result = optimizer.run(side='long', objective='calmar')

print(f'\nOptimization complete: {result.n_trials} trials in {result.elapsed_secs:.1f}s')
print(f'Best score:   {result.best_score:.4f}')
print(f'Best params:  {result.best_params}')

# Show top 5
print('\nTop 5 parameter sets:')
top5 = result.top_n(5)
for i, r in enumerate(top5, 1):
    print(f'  {i}. score={r["score"]:.4f}  params={r["params"]}')

## 10. Walk-Forward Validation

In [ ]:
from validation.walk_forward import WalkForwardValidator

wf = WalkForwardValidator(
    n_folds=4,
    purge_gap_bars=10,
    expanding=True,
)

strategy = TrendBreakoutStrategy()
print('Running walk-forward validation (4 folds, 50 trials each)...')
wf_result = wf.run(
    strategy=strategy,
    df=df_eur,
    side='long',
    objective='calmar',
    opt_method='optuna',
    opt_trials=50,
    min_trades=10,
)

print(wf_result.summary())

## 11. Performance by Regime

In [ ]:
from validation.metrics import compute_metrics_by_group

if not wf_result.combined_trades.empty:
    trades_with_regime = wf_result.combined_trades.copy()
    # Attach regime (approximate by entry time)
    if 'entry_time' in trades_with_regime.columns:
        regime_at_entry = regime_df['smoothed_regime'].reindex(trades_with_regime['entry_time'].values, method='nearest')
        trades_with_regime['regime'] = regime_at_entry.values
        
        by_regime = compute_metrics_by_group(trades_with_regime, 'regime')
        print('Performance by regime:')
        print(by_regime[['n_trades','win_rate','profit_factor','total_return_pct']].to_string())
        
        from utils.plotting import plot_performance_by_regime
        if 'profit_factor' in by_regime.columns:
            fig = plot_performance_by_regime(by_regime, metric='profit_factor')
            plt.show()
else:
    print('No walk-forward trades available for regime breakdown.')

## 12. Full Signal Pipeline (Regime → Signal)

In [ ]:
from selection.regime_strategy_router import RegimeStrategyRouter
from filters.tradability_filter import TradabilityFilter
from optimization.parameter_store import ParameterStore
from execution.signal_engine import SignalEngine
from portfolio.risk_engine import RiskEngine

# Build market data dict with features
market_data = {
    'EURUSD': df_eur,
    'USDJPY': df_jpy,
    'XAUUSD': df_xau,
}

# Add features to JPY and XAU
market_data['USDJPY'] = add_alpha_features(market_data['USDJPY'])
market_data['XAUUSD'] = add_alpha_features(market_data['XAUUSD'])

# Router
tf_filter   = TradabilityFilter()
param_store = ParameterStore()

router = RegimeStrategyRouter(
    regime_service=regime_svc,
    parameter_store=param_store,
    asset_universe=['EURUSD', 'USDJPY', 'XAUUSD'],
    tradability_filter=tf_filter,
)

print('Running signal pipeline...')
proposals = router.route(market_data)
print(f'Proposals generated: {len(proposals)}')

for p in proposals:
    print(f'  {p.asset:8s} {p.side:5s} {p.strategy:20s} | '
          f'entry={p.entry:.5f} sl={p.stop_loss:.5f} tp={p.take_profit:.5f} '
          f'RR={p.rr_ratio:.2f} | regime={p.regime}')

## 13. Risk Approval & MT5 Signal Output

In [ ]:
import json
from execution.signal_engine import SignalEngine
from portfolio.risk_engine import RiskEngine

risk_engine = RiskEngine(
    risk_pct=0.5,
    max_open_trades=3,
    min_rr_ratio=1.5,
)

engine = SignalEngine(
    risk_engine=risk_engine,
    paper_mode=True,
)

signals = engine.process(proposals, equity=10000.0)

if signals:
    print(f'Approved signals: {len(signals)}')
    for sig in signals:
        print(json.dumps(sig.to_dict(), indent=2))
    # Write the best signal to JSON for MT5
    from pathlib import Path
    output_path = Path('../signals/latest_signal.json')
    signals[0].save(output_path)
    print(f'\nSignal saved to {output_path}')
else:
    print('No signals approved (check regime conditions or risk limits)')
    engine.write_no_signal()

## 14. Summary

The framework has demonstrated:

1. **Real data loading** via yfinance (no API key needed)
2. **25 regime and alpha features** computed in one pass
3. **Ensemble regime detection** (HMM + LightGBM, 7 regime classes)
4. **Session and liquidity filtering** (London + NY sessions)
5. **4 independent strategies** with separate long/short parameters
6. **Vectorized backtesting** with full trade simulation
7. **Bayesian parameter optimization** (Optuna, 50+ trials)
8. **Walk-forward validation** (4 folds, expanding window, purge gap)
9. **Full signal pipeline** (regime → asset/strategy/side selection → proposal)
10. **Risk-gated MT5-compatible JSON signal output**

---

### Next Steps

- Run `scripts/run_optimization.py` with `--trials 200 --folds 5` to find robust parameters
- Load the saved parameters from `parameters/parameter_store.json` into the router
- Schedule `scripts/generate_signal.py` hourly via Task Scheduler
- Set `MT5_SIGNAL_OUTPUT_PATH` to the MT5 Files/ directory
- Deploy the MT5 bridge EA to read and execute the signals
